In [ ]:
FIGURE = 'Fig.S1'
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import gc
import mmap
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import netCDF4 as nc
_NETCDF_DATASET = nc.Dataset
HERE = Path.cwd().resolve()
if not (HERE / 'Inputs').is_dir():
    for candidate in [HERE / 'Figure' / FIGURE, HERE / FIGURE, HERE / 'Data Processing' if FIGURE in ['SSMIrz','Rmax_Ts'] else HERE]:
        if (candidate / 'Inputs').is_dir():
            HERE = candidate
            break
INPUT_DIR = HERE / 'Inputs'
OUTPUT_DIR = HERE / 'Outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
ROOT = HERE
DATA = INPUT_DIR
OUTPUT = OUTPUT_DIR
__file__ = str(HERE / (FIGURE + '.ipynb'))
sys.argv = ['notebook']

def open_nc(path, mode='r', **kwargs):
    if 'memory' in kwargs:
        return _NETCDF_DATASET(path, mode, **kwargs)
    path = Path(path).resolve()
    previous = Path.cwd()
    try:
        os.chdir(path.parent)
        return _NETCDF_DATASET(path.name, mode, **kwargs)
    finally:
        os.chdir(previous)

def asset_path(key):
    if key in ['asset_e2603c619a67', 'asset_5e3c32038bfb']:
        return INPUT_DIR / 'study_mask.tif'
    if key == 'asset_beb53c9b3525':
        return INPUT_DIR / 'SPEI_k01-k24.nc'
    if key == 'asset_174bc324d489':
        return INPUT_DIR / 'SSMIrz_k01-k24.nc'
    return OUTPUT_DIR

def output_path(name=''):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path

def module_path(name=''):
    return str(HERE)

def font_path(filename):
    from matplotlib.font_manager import FontProperties, findfont
    family = 'Times New Roman' if filename.lower().startswith('times') else 'Courier New' if filename.lower().startswith('cour') else 'Arial'
    return findfont(FontProperties(family=family))

def array(value):
    return np.asarray(np.ma.filled(value.astype(float), np.nan))

@contextmanager
def mapped(path):
    with Path(path).open('rb') as stream:
        mm = mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ)
        dataset = open_nc('input.nc', memory=mm)
        try:
            yield dataset
        finally:
            dataset.close()
            gc.collect()
            mm.close()

class _QuadrantBase:
    asset_path = asset_path
    module_path = module_path
    output_path = output_path
    font_path = font_path
    import json
    import os
    import shutil
    from datetime import datetime
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    from PIL import Image
    import matplotlib.colors as mcolors
    import matplotlib.pyplot as plt
    import netCDF4 as nc
    import numpy as np
    import pandas as pd
    ASCII_ROOT = asset_path('asset_4d8b1bf7a531')
    ASCII_OUT = asset_path('asset_f7d6ecaa36de')
    FINAL_ROOT = asset_path('asset_1356a3b3c5bd')
    PATHWAYS = ['SPEI', 'SSMIrz']
    SEASONS = ['MAM', 'JJA', 'SON', 'DJF']
    Q_SIG = 0.05
    RMAX_FILES = {'SPEI': os.path.join(ASCII_ROOT, '01_rmax_trend_all_valid', 'SIF_SPEI_maxr_theilsen_trend_all_valid.nc'), 'SSMIrz': os.path.join(ASCII_ROOT, '01_rmax_trend_all_valid', 'SIF_SSMIrz_maxr_theilsen_trend_all_valid.nc')}
    TS_FILES = {'SPEI': os.path.join(ASCII_ROOT, '02_optscale_trend_jump_filter', 'SIF_SPEI_optscale_theilsen_trend_jump_filter.nc'), 'SSMIrz': os.path.join(ASCII_ROOT, '02_optscale_trend_jump_filter', 'SIF_SSMIrz_optscale_theilsen_trend_jump_filter.nc')}
    QUADRANTS = [('$T_s^{+},\\ R_{\\max}^{+}$', lambda X, Y: (X > 0) & (Y > 0), 'pos_pos'), ('$T_s^{-},\\ R_{\\max}^{+}$', lambda X, Y: (X < 0) & (Y > 0), 'neg_pos'), ('$T_s^{-},\\ R_{\\max}^{-}$', lambda X, Y: (X < 0) & (Y < 0), 'neg_neg'), ('$T_s^{+},\\ R_{\\max}^{-}$', lambda X, Y: (X > 0) & (Y < 0), 'pos_neg')]
    ENDPOINTS = {'pos_pos': {'light': '#D4D5F8', 'dark': '#626FB3'}, 'pos_neg': {'light': '#B7E6FA', 'dark': '#059ECC'}, 'neg_pos': {'light': '#FFB9B9FF', 'dark': '#FF0000FF'}, 'neg_neg': {'light': '#D6E6CB', 'dark': '#1DA51D'}}

    def setup_reference_style():
        _QuadrantBase.plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans', 'Arial', 'Liberation Sans']
        _QuadrantBase.plt.rcParams['axes.unicode_minus'] = False
        _QuadrantBase.plt.rcParams['font.family'] = 'serif'
        _QuadrantBase.plt.rcParams['font.serif'] = ['Arial', 'DejaVu Serif', 'Liberation Serif']
        _QuadrantBase.plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
        _QuadrantBase.plt.rcParams['font.size'] = 12
        _QuadrantBase.plt.rcParams['axes.titlesize'] = 14
        _QuadrantBase.plt.rcParams['axes.labelsize'] = 12
        _QuadrantBase.plt.rcParams['xtick.labelsize'] = 10
        _QuadrantBase.plt.rcParams['ytick.labelsize'] = 10
        _QuadrantBase.plt.rcParams['legend.fontsize'] = 10
        _QuadrantBase.plt.rcParams['figure.titlesize'] = 16
        _QuadrantBase.plt.rcParams['figure.dpi'] = 100
        _QuadrantBase.plt.rcParams['savefig.dpi'] = 600

    def filled(x, fill_value=np.nan):
        if _QuadrantBase.np.ma.isMaskedArray(x):
            return x.filled(fill_value)
        return _QuadrantBase.np.asarray(x)

    def read_season_names(ds):
        return [str(x) for x in _QuadrantBase.nc.chartostring(ds.variables['season_name'][:])]

    def sort_lon(lon, data):
        lon = _QuadrantBase.np.asarray(lon, dtype=_QuadrantBase.np.float64)
        lon2 = (lon + 180.0) % 360.0 - 180.0
        order = _QuadrantBase.np.argsort(lon2)
        arr = _QuadrantBase.np.asarray(data)
        if arr.ndim >= 3 and arr.shape[-1] == 3 and (arr.shape[-2] == len(lon)):
            return (lon2[order], arr[..., order, :])
        return (lon2[order], arr[..., order])

    def calculate_latitude_weights(lat):
        lat_rad = _QuadrantBase.np.deg2rad(lat)
        return _QuadrantBase.np.abs(_QuadrantBase.np.cos(lat_rad))

    def create_weight_matrix(lon, lat):
        lat_weights = _QuadrantBase.calculate_latitude_weights(lat)
        return _QuadrantBase.np.broadcast_to(lat_weights[:, _QuadrantBase.np.newaxis], (len(lat), len(lon)))

    def to_rgb(c):
        if isinstance(c, str):
            return _QuadrantBase.np.array(_QuadrantBase.mcolors.to_rgb(c), dtype=float)
        return _QuadrantBase.np.array(c, dtype=float)

    def bivariate_cmap_endpoints(X, Y, endpoints, gamma=1.0, robust=95, nodata_color=(1, 1, 1)):
        valid = ~(_QuadrantBase.np.isnan(X) | _QuadrantBase.np.isnan(Y))
        X0 = _QuadrantBase.np.where(valid, X, 0.0)
        Y0 = _QuadrantBase.np.where(valid, Y, 0.0)
        Xv = X0[valid]
        Yv = Y0[valid]
        if Xv.size > 0:
            X_max = _QuadrantBase.np.percentile(_QuadrantBase.np.abs(Xv), robust)
            Y_max = _QuadrantBase.np.percentile(_QuadrantBase.np.abs(Yv), robust)
            X_max = max(X_max, 1e-12)
            Y_max = max(Y_max, 1e-12)
        else:
            X_max = Y_max = 1.0
        Xn = _QuadrantBase.np.clip(X0 / X_max, -1, 1)
        Yn = _QuadrantBase.np.clip(Y0 / Y_max, -1, 1)
        img = _QuadrantBase.np.ones(X.shape + (3,), dtype=float)

        def paint(mask2, key):
            if not _QuadrantBase.np.any(mask2):
                return
            strength = _QuadrantBase.np.sqrt(Xn[mask2] ** 2 + Yn[mask2] ** 2) / _QuadrantBase.np.sqrt(2)
            strength = _QuadrantBase.np.clip(strength, 0, 1)
            if gamma != 1.0:
                strength = strength ** gamma
            light = _QuadrantBase.to_rgb(endpoints[key]['light'])
            dark = _QuadrantBase.to_rgb(endpoints[key]['dark'])
            img[mask2] = (1 - strength)[:, None] * light + strength[:, None] * dark
        paint((Xn >= 0) & (Yn >= 0) & valid, 'pos_pos')
        paint((Xn >= 0) & (Yn < 0) & valid, 'pos_neg')
        paint((Xn < 0) & (Yn >= 0) & valid, 'neg_pos')
        paint((Xn < 0) & (Yn < 0) & valid, 'neg_neg')
        img[~valid] = _QuadrantBase.np.array(nodata_color, dtype=float)
        return _QuadrantBase.np.clip(img, 0, 1)

    def weighted_mean(values, weights):
        v = _QuadrantBase.np.asarray(values, dtype=float).ravel()
        w = _QuadrantBase.np.asarray(weights, dtype=float).ravel()
        good = ~_QuadrantBase.np.isnan(v) & ~_QuadrantBase.np.isnan(w) & (w > 0)
        if not _QuadrantBase.np.any(good):
            return _QuadrantBase.np.nan
        return _QuadrantBase.np.sum(v[good] * w[good]) / _QuadrantBase.np.sum(w[good])

    def add_significance_stipple_points(ax, lon, lat, X_sig, Y_sig, step=6, s=10, s_both=16, alpha=0.7, ts_color='0.55', mc_color='#7B2CBF', both_color='k', zorder=6):
        Lon, Lat = _QuadrantBase.np.meshgrid(lon, lat)
        sl = (slice(None, None, int(step)), slice(None, None, int(step)))
        Lon_s = Lon[sl]
        Lat_s = Lat[sl]
        Xs = X_sig[sl]
        Ys = Y_sig[sl]
        both = Xs & Ys
        only_x = Xs & ~Ys
        only_y = Ys & ~Xs
        if _QuadrantBase.np.any(only_x):
            ax.scatter(Lon_s[only_x], Lat_s[only_x], s=s, c=ts_color, marker='.', alpha=alpha, linewidths=0, transform=_QuadrantBase.ccrs.PlateCarree(), zorder=zorder)
        if _QuadrantBase.np.any(only_y):
            ax.scatter(Lon_s[only_y], Lat_s[only_y], s=s, c=mc_color, marker='.', alpha=alpha, linewidths=0, transform=_QuadrantBase.ccrs.PlateCarree(), zorder=zorder)
        if _QuadrantBase.np.any(both):
            ax.scatter(Lon_s[both], Lat_s[both], s=s_both, c=both_color, marker='.', alpha=min(0.9, alpha + 0.1), linewidths=0, transform=_QuadrantBase.ccrs.PlateCarree(), zorder=zorder + 1)

    def _fixed_quadrant_label(ax, row, key):
        from matplotlib.offsetbox import TextArea, DrawingArea, HPacker, AnnotationBbox
        from matplotlib.lines import Line2D
        fontsize = 8.5
        props = dict(fontsize=fontsize, fontfamily='Arial', color='black')
        if key == 'zero':
            label = TextArea('either = 0', textprops=props)
        else:
            signs = {'pos_pos': ('+', '+'), 'neg_pos': ('-', '+'), 'neg_neg': ('-', '-'), 'pos_neg': ('+', '-')}[key]

            def symbol(sign):
                cell = DrawingArea(5.5, 9, 0, 0)
                cell.add_artist(Line2D([0.8, 4.7], [3.0, 3.0], color='black', linewidth=0.6))
                if sign == '+':
                    cell.add_artist(Line2D([2.75, 2.75], [1.05, 4.95], color='black', linewidth=0.6))
                return cell
            label = HPacker(children=[TextArea('Ts', textprops=props), symbol(signs[0]), TextArea(', ', textprops=props), TextArea('$R_{\\max}$', textprops=props), symbol(signs[1])], align='baseline', pad=0, sep=0)
        artist = AnnotationBbox(label, (0, row), xycoords=ax.get_yaxis_transform(), xybox=(-7, 0), boxcoords='offset points', box_alignment=(1, 0.5), frameon=False, pad=0, annotation_clip=False)
        artist.set_zorder(50)
        ax.add_artist(artist)
        return artist

    def add_weighted_significance_bar_chart(fig, X_slope_masked, Y_slope_masked, X_pvalue_masked, Y_pvalue_masked, weight_matrix, mask, endpoints, position=(0.05, 0.05), width=0.18, height=0.4, show_legend=True, annotate_total=True, annotate_sig=False):
        valid_mask = mask & ~_QuadrantBase.np.isnan(X_slope_masked) & ~_QuadrantBase.np.isnan(Y_slope_masked)
        if not _QuadrantBase.np.any(valid_mask):
            return (None, None)
        total_weight = _QuadrantBase.np.sum(weight_matrix[valid_mask])
        if total_weight <= 0:
            return (None, None)
        X_sig = (X_pvalue_masked < _QuadrantBase.Q_SIG) & ~_QuadrantBase.np.isnan(X_pvalue_masked)
        Y_sig = (Y_pvalue_masked < _QuadrantBase.Q_SIG) & ~_QuadrantBase.np.isnan(Y_pvalue_masked)
        both_sig = X_sig & Y_sig
        quadrant_data = []
        for name, cond, ckey in _QuadrantBase.QUADRANTS:
            q_mask = cond(X_slope_masked, Y_slope_masked) & valid_mask
            total_q_weight = _QuadrantBase.np.sum(weight_matrix[q_mask]) if _QuadrantBase.np.any(q_mask) else 0.0
            sig_q_weight = _QuadrantBase.np.sum(weight_matrix[q_mask & both_sig]) if _QuadrantBase.np.any(q_mask & both_sig) else 0.0
            nonsig_q_weight = total_q_weight - sig_q_weight
            quadrant_data.append({'name': name, 'color_key': ckey, 'total_pct': total_q_weight / total_weight * 100, 'sig_pct': sig_q_weight / total_weight * 100, 'nonsig_pct': nonsig_q_weight / total_weight * 100})
        ax_bar = fig.add_axes([position[0], position[1], width, height])
        y_pos = _QuadrantBase.np.arange(len(quadrant_data))
        ax_bar.barh(y_pos, [q['total_pct'] for q in quadrant_data], color=[endpoints[q['color_key']]['dark'] for q in quadrant_data], alpha=0.95, edgecolor='white', linewidth=1)
        ax_bar.set_yticks(y_pos)
        ax_bar.set_yticklabels([''] * len(quadrant_data))
        ax_bar.set_xlabel('', fontsize=9)
        ax_bar.set_title('', fontsize=11, weight='bold', pad=6)
        if annotate_total:
            for i, q in enumerate(quadrant_data):
                if q['name'] == '$T_s^{-},\\ R_{\\max}^{-}$':
                    total_x_pos = q['total_pct'] + 2.0
                elif q['name'] == '$T_s^{-},\\ R_{\\max}^{+}$':
                    total_x_pos = q['total_pct'] + 1.0
                elif q['total_pct'] < 15:
                    total_x_pos = q['total_pct'] + 0.8
                else:
                    total_x_pos = q['total_pct'] + 0.5
                ax_bar.text(total_x_pos, i, f"{q['total_pct']:.1f}%", ha='left', va='center', fontsize=8.5, weight='bold', color='0.1')
        max_pct = max((q['total_pct'] for q in quadrant_data)) if quadrant_data else 1
        ax_bar.set_xlim(0, max_pct * 1.6)
        ax_bar.spines['top'].set_visible(False)
        ax_bar.spines['right'].set_visible(False)
        ax_bar.grid(axis='x', alpha=0.25, linestyle='--')
        ax_bar.tick_params(axis='x', labelsize=8)
        ax_bar.tick_params(axis='y', labelsize=9, pad=0.5)
        ax_bar.invert_yaxis()
        for row, q in enumerate(quadrant_data):
            _QuadrantBase._fixed_quadrant_label(ax_bar, row, q['color_key'])
        return (ax_bar, quadrant_data)

    def add_dual_axis_lollipop_inset(fig, X_slope_masked, Y_slope_masked, X_pvalue_masked, Y_pvalue_masked, weight_matrix, mask, endpoints, position=(0.76, 0.02), width=0.22, height=0.18, use_weighted_mean=True, ts_range=(-1.8, 1.8), ts_tick_value=1.0, mc_tick_value=0.03, show_both_sig=False):
        ax = fig.add_axes([position[0], position[1], width, height])
        valid_mask = mask & ~_QuadrantBase.np.isnan(X_slope_masked) & ~_QuadrantBase.np.isnan(Y_slope_masked)
        if not _QuadrantBase.np.any(valid_mask):
            ax.set_axis_off()
            return (ax, None)
        X_sig = (X_pvalue_masked < _QuadrantBase.Q_SIG) & ~_QuadrantBase.np.isnan(X_pvalue_masked)
        Y_sig = (Y_pvalue_masked < _QuadrantBase.Q_SIG) & ~_QuadrantBase.np.isnan(Y_pvalue_masked)
        both_sig = X_sig & Y_sig
        rows = []
        for name, cond, ckey in _QuadrantBase.QUADRANTS:
            qmask = cond(X_slope_masked, Y_slope_masked) & valid_mask
            if not _QuadrantBase.np.any(qmask):
                ts_stat = _QuadrantBase.np.nan
                mc_stat = _QuadrantBase.np.nan
                ts_stat_sig = _QuadrantBase.np.nan
                mc_stat_sig = _QuadrantBase.np.nan
            else:
                ts_stat = _QuadrantBase.weighted_mean(X_slope_masked[qmask], weight_matrix[qmask])
                mc_stat = _QuadrantBase.weighted_mean(Y_slope_masked[qmask], weight_matrix[qmask])
                if show_both_sig:
                    qs = qmask & both_sig
                    ts_stat_sig = _QuadrantBase.weighted_mean(X_slope_masked[qs], weight_matrix[qs]) if _QuadrantBase.np.any(qs) else _QuadrantBase.np.nan
                    mc_stat_sig = _QuadrantBase.weighted_mean(Y_slope_masked[qs], weight_matrix[qs]) if _QuadrantBase.np.any(qs) else _QuadrantBase.np.nan
                else:
                    ts_stat_sig = _QuadrantBase.np.nan
                    mc_stat_sig = _QuadrantBase.np.nan
            rows.append({'name': name, 'color_key': ckey, 'ts_mean': ts_stat, 'mc_mean': mc_stat, 'ts_mean_sig': ts_stat_sig, 'mc_mean_sig': mc_stat_sig})
        ax.set_ylim(ts_range)
        ax.axhline(0, color='0.25', lw=1)
        ax.set_yticks([-ts_tick_value, ts_tick_value])
        ax.set_yticklabels([f'{-ts_tick_value:.0f}', f'{ts_tick_value:.0f}'])
        ax2 = ax.twinx()
        scale_factor = ts_range[1] / ts_tick_value
        mc_range_max = mc_tick_value
        mc_range = (-mc_range_max, mc_range_max)
        ax2.set_ylim(mc_range)
        ax2.axhline(0, color='0.25', lw=1)
        ax2.set_yticks([-mc_tick_value, mc_tick_value])
        ax2.set_yticklabels([f'{-mc_tick_value:.2f}', f'{mc_tick_value:.2f}'])
        x = _QuadrantBase.np.arange(len(rows))
        ax.set_xticks([])
        ax.set_xticklabels([])
        dx = 0.14
        for i, r in enumerate(rows):
            xi = x[i]
            c_dark = endpoints[r['color_key']]['dark']
            c_light = endpoints[r['color_key']]['light']
            if _QuadrantBase.np.isfinite(r['ts_mean']):
                ax.vlines(xi - dx, 0, r['ts_mean'], color=c_dark, lw=2.0, alpha=0.95)
                ax.scatter([xi - dx], [r['ts_mean']], s=34, color=c_dark, edgecolor='white', linewidth=0.6, zorder=5)
            if _QuadrantBase.np.isfinite(r['mc_mean']):
                ax2.vlines(xi + dx, 0, r['mc_mean'], color=c_light, lw=2.0, alpha=0.95)
                ax2.scatter([xi + dx], [r['mc_mean']], s=34, color=c_light, edgecolor='white', linewidth=0.6, zorder=5)
            if show_both_sig and _QuadrantBase.np.isfinite(r['ts_mean_sig']):
                ax.scatter([xi - dx], [r['ts_mean_sig']], s=48, facecolors='none', edgecolors='k', linewidth=1.0, zorder=6)
            if show_both_sig and _QuadrantBase.np.isfinite(r['mc_mean_sig']):
                ax2.scatter([xi + dx], [r['mc_mean_sig']], s=48, facecolors='none', edgecolors='k', linewidth=1.0, zorder=6)
        ax.grid(axis='y', alpha=0.25, linestyle='--')
        ax.spines['top'].set_visible(True)
        ax.spines['bottom'].set_visible(True)
        ax.spines['left'].set_visible(True)
        ax.spines['right'].set_visible(True)
        for spine in ax.spines.values():
            spine.set_linewidth(0.8)
            spine.set_color('black')
        ax.tick_params(axis='y', labelsize=8, left=True, labelleft=True, right=False, labelright=False)
        ax2.tick_params(axis='y', labelsize=8, left=False, labelleft=False, right=True, labelright=True)
        ax.set_ylabel('Ts', fontsize=9, color='0.2', rotation=0, labelpad=15)
        ax.yaxis.set_label_coords(-0.2, 0.4)
        ax2.set_ylabel('$R_{\\max}$', fontsize=9, color='0.2', rotation=0, labelpad=15)
        ax2.yaxis.set_label_coords(1.45, 0.5)
        ax2.yaxis.label.set_rotation(0)
        ax2.yaxis.label.set_bbox(dict(facecolor='white', edgecolor='none', pad=1.2))
        ax2.yaxis.label.set_zorder(100)
        ax.yaxis.label.set_bbox(dict(facecolor='white', edgecolor='none', pad=1.2))
        ax.set_title('Trend slope', weight='bold', fontsize=9, pad=6)
        ax.set_xlim(-0.5, len(rows) - 0.5)
        return (ax, rows)

    def read_panel(pathway, season):
        with _QuadrantBase.nc.Dataset(_QuadrantBase.TS_FILES[pathway]) as ts_ds, _QuadrantBase.nc.Dataset(_QuadrantBase.RMAX_FILES[pathway]) as r_ds:
            lat = _QuadrantBase.np.asarray(ts_ds.variables['lat'][:])
            lon = _QuadrantBase.np.asarray(ts_ds.variables['lon'][:])
            seasons = _QuadrantBase.read_season_names(ts_ds)
            si = {s: i for i, s in enumerate(seasons)}[season]
            ts = _QuadrantBase.filled(ts_ds.variables['slope'][si], _QuadrantBase.np.nan).astype(float)
            rmax = _QuadrantBase.filled(r_ds.variables['slope'][si], _QuadrantBase.np.nan).astype(float)
            q_ts = _QuadrantBase.filled(ts_ds.variables['q_fdr'][si], _QuadrantBase.np.nan).astype(float)
            q_r = _QuadrantBase.filled(r_ds.variables['q_fdr'][si], _QuadrantBase.np.nan).astype(float)
            mask = (_QuadrantBase.filled(ts_ds.variables['analysis_mask'][si], 0).astype(_QuadrantBase.np.uint8) == 1) & (_QuadrantBase.filled(r_ds.variables['analysis_mask'][si], 0).astype(_QuadrantBase.np.uint8) == 1)
        return (lat, lon, ts, rmax, q_ts, q_r, mask)

    def panel_statistics(pathway, season, lat, lon, ts, rmax, q_ts, q_r, mask, quadrant_data, lolli_rows):
        weights = _QuadrantBase.create_weight_matrix(lon, lat)
        valid = mask & _QuadrantBase.np.isfinite(ts) & _QuadrantBase.np.isfinite(rmax)
        total_w = _QuadrantBase.np.sum(weights[valid])
        rows = []
        for q in quadrant_data:
            rows.append({'pathway': pathway, 'season': season, 'quadrant': q['name'], 'area_percent': q['total_pct'], 'both_qfdr_sig_area_percent': q['sig_pct']})
        if lolli_rows:
            for r in lolli_rows:
                rows.append({'pathway': pathway, 'season': season, 'quadrant': r['name'], 'ts_weighted_mean': r['ts_mean'], 'rmax_weighted_mean': r['mc_mean']})
        rows.append({'pathway': pathway, 'season': season, 'quadrant': 'SIGNIFICANCE_SUMMARY', 'ts_qfdr_sig_area_percent': float(_QuadrantBase.np.sum(weights[valid & (q_ts < _QuadrantBase.Q_SIG)]) / total_w * 100) if total_w > 0 else _QuadrantBase.np.nan, 'rmax_qfdr_sig_area_percent': float(_QuadrantBase.np.sum(weights[valid & (q_r < _QuadrantBase.Q_SIG)]) / total_w * 100) if total_w > 0 else _QuadrantBase.np.nan, 'both_qfdr_sig_area_percent': float(_QuadrantBase.np.sum(weights[valid & (q_ts < _QuadrantBase.Q_SIG) & (q_r < _QuadrantBase.Q_SIG)]) / total_w * 100) if total_w > 0 else _QuadrantBase.np.nan})
        return rows

    def draw_single_original_layout(pathway, season):
        lat, lon, ts, rmax, q_ts, q_r, mask = _QuadrantBase.read_panel(pathway, season)
        lon_raw = lon.copy()
        lon, ts = _QuadrantBase.sort_lon(lon_raw, ts)
        _, rmax = _QuadrantBase.sort_lon(lon_raw, rmax)
        _, q_ts = _QuadrantBase.sort_lon(lon_raw, q_ts)
        _, q_r = _QuadrantBase.sort_lon(lon_raw, q_r)
        _, mask = _QuadrantBase.sort_lon(lon_raw, mask.astype(_QuadrantBase.np.uint8))
        mask = mask.astype(bool)
        X_slope_masked = ts.copy()
        Y_slope_masked = rmax.copy()
        X_pvalue_masked = q_ts.copy()
        Y_pvalue_masked = q_r.copy()
        X_slope_masked[~mask] = _QuadrantBase.np.nan
        Y_slope_masked[~mask] = _QuadrantBase.np.nan
        X_pvalue_masked[~mask] = _QuadrantBase.np.nan
        Y_pvalue_masked[~mask] = _QuadrantBase.np.nan
        X_significant = (X_pvalue_masked < _QuadrantBase.Q_SIG) & ~_QuadrantBase.np.isnan(X_pvalue_masked) & mask
        Y_significant = (Y_pvalue_masked < _QuadrantBase.Q_SIG) & ~_QuadrantBase.np.isnan(Y_pvalue_masked) & mask
        weight_matrix = _QuadrantBase.create_weight_matrix(lon, lat)
        fig = _QuadrantBase.plt.figure(figsize=(7, 4))
        ax = _QuadrantBase.plt.axes(projection=_QuadrantBase.ccrs.PlateCarree())
        img = _QuadrantBase.bivariate_cmap_endpoints(X_slope_masked, Y_slope_masked, endpoints=_QuadrantBase.ENDPOINTS, gamma=1.0, robust=95, nodata_color=(1, 1, 1))
        dlon = abs(lon[1] - lon[0]) if len(lon) > 1 else 0.1
        dlat = abs(lat[1] - lat[0]) if len(lat) > 1 else 0.1
        ax.imshow(img, origin='lower', extent=[lon.min(), lon.max(), lat.min(), lat.max()], transform=_QuadrantBase.ccrs.PlateCarree(), interpolation='nearest', zorder=1)
        ax.coastlines('110m', linewidth=0.8, color='black', zorder=10)
        ax.add_feature(_QuadrantBase.cfeature.BORDERS, linewidth=0.5, color='gray', zorder=10)
        ax.add_feature(_QuadrantBase.cfeature.LAND, facecolor='lightgray', alpha=0.2, zorder=0)
        ax.add_feature(_QuadrantBase.cfeature.OCEAN, facecolor='lightblue', alpha=0.1, zorder=0)
        ax.set_extent([lon.min() - dlon * 0.5, lon.max() + dlon * 0.5, lat.min() - dlat * 0.5, lat.max() + dlat * 0.5], crs=_QuadrantBase.ccrs.PlateCarree())
        gl = ax.gridlines(crs=_QuadrantBase.ccrs.PlateCarree(), draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {'size': 13, 'family': 'serif'}
        gl.ylabel_style = {'size': 13, 'family': 'serif'}
        ax_bar, quadrant_data = _QuadrantBase.add_weighted_significance_bar_chart(fig, X_slope_masked, Y_slope_masked, X_pvalue_masked, Y_pvalue_masked, weight_matrix, mask, _QuadrantBase.ENDPOINTS, position=(0.05, 0.05), width=0.18, height=0.4, show_legend=True, annotate_total=True, annotate_sig=False)
        if ax_bar is not None:
            ax_bar.set_position([0.232, 0.28, 0.09, 0.25])
        ax_lolli, lolli_rows = _QuadrantBase.add_dual_axis_lollipop_inset(fig, X_slope_masked, Y_slope_masked, X_pvalue_masked, Y_pvalue_masked, weight_matrix, mask, _QuadrantBase.ENDPOINTS, position=(0.62, 0.06), width=0.2, height=0.16, use_weighted_mean=True, ts_range=(-1.8, 1.8), ts_tick_value=1.0, mc_tick_value=0.03, show_both_sig=False)
        if ax_lolli is not None:
            ax_lolli.set_position([0.58, 0.18, 0.2, 0.15])
        single_dir = _QuadrantBase.os.path.join(_QuadrantBase.ASCII_OUT, 'single_panels')
        _QuadrantBase.os.makedirs(single_dir, exist_ok=True)
        png = _QuadrantBase.os.path.join(single_dir, f'fourcolor_original_layout_{pathway}_{season}_600dpi.png')
        tif = _QuadrantBase.os.path.join(single_dir, f'fourcolor_original_layout_{pathway}_{season}_600dpi.tif')
        pdf = _QuadrantBase.os.path.join(single_dir, f'fourcolor_original_layout_{pathway}_{season}.pdf')
        fig.savefig(png, dpi=600, bbox_inches='tight', pad_inches=0.2, facecolor='white', edgecolor='none', format='png', metadata={'Software': 'matplotlib'})
        fig.savefig(tif, dpi=600, bbox_inches='tight', pad_inches=0.2, facecolor='white', edgecolor='none')
        fig.savefig(pdf, bbox_inches='tight', pad_inches=0.2, facecolor='white', edgecolor='none')
        _QuadrantBase.plt.close(fig)
        stats_rows = _QuadrantBase.panel_statistics(pathway, season, lat, lon, X_slope_masked, Y_slope_masked, X_pvalue_masked, Y_pvalue_masked, mask, quadrant_data, lolli_rows)
        return {'png': png, 'tif': tif, 'pdf': pdf, 'stats': stats_rows}

    def stitch_panels(panel_pngs):
        images = [_QuadrantBase.Image.open(p).convert('RGB') for p in panel_pngs]
        w = max((im.width for im in images))
        h = max((im.height for im in images))
        canvas = _QuadrantBase.Image.new('RGB', (w * 2, h * 4), 'white')
        for idx, im in enumerate(images):
            row = idx // 2
            col = idx % 2
            x = col * w + (w - im.width) // 2
            y = row * h + (h - im.height) // 2
            canvas.paste(im, (x, y))
        out_png = _QuadrantBase.os.path.join(_QuadrantBase.ASCII_OUT, 'FigS_fourcolor_Ts_Rmax_original_layout_8panels_600dpi.png')
        out_tif = _QuadrantBase.os.path.join(_QuadrantBase.ASCII_OUT, 'FigS_fourcolor_Ts_Rmax_original_layout_8panels_600dpi.tif')
        canvas.save(out_png, dpi=(600, 600))
        canvas.save(out_tif, dpi=(600, 600))
        return (out_png, out_tif)

    def main():
        _QuadrantBase.setup_reference_style()
        _QuadrantBase.os.makedirs(_QuadrantBase.ASCII_OUT, exist_ok=True)
        _QuadrantBase.os.makedirs(_QuadrantBase.FINAL_ROOT, exist_ok=True)
        panel_order = [(pathway, season) for season in _QuadrantBase.SEASONS for pathway in _QuadrantBase.PATHWAYS]
        outputs = {}
        stats_rows = []
        panel_pngs = []
        for pathway, season in panel_order:
            res = _QuadrantBase.draw_single_original_layout(pathway, season)
            outputs[f'{pathway}_{season}'] = {k: v for k, v in res.items() if k != 'stats'}
            stats_rows.extend(res['stats'])
            panel_pngs.append(res['png'])
        stitched_png, stitched_tif = _QuadrantBase.stitch_panels(panel_pngs)
        stats_csv = _QuadrantBase.os.path.join(_QuadrantBase.ASCII_OUT, 'fourcolor_Ts_Rmax_original_layout_8panels_stats.csv')
        _QuadrantBase.pd.DataFrame(stats_rows).to_csv(stats_csv, index=False, encoding='utf-8-sig')
        log_path = _QuadrantBase.os.path.join(_QuadrantBase.ASCII_OUT, '24_quadrant_map_original_layout_retained_8panel_mosaic_log.json')
        with open(log_path, 'w', encoding='utf-8') as f:
            _QuadrantBase.json.dump({'created_at': _QuadrantBase.datetime.now().isoformat(timespec='seconds'), 'layout': 'Single-panel layout follows the reference notebook without changing positions.', 'panel_order': [f'{p}_{s}' for p, s in panel_order], 'classification': [q[0] for q in _QuadrantBase.QUADRANTS], 'significance': 'q_fdr < 0.05 based on p_eff after effective-dof correction; no stippling is drawn on maps.', 'outputs': outputs, 'stitched': {'png': stitched_png, 'tif': stitched_tif}, 'stats': stats_csv}, f, ensure_ascii=False, indent=2)
        final_single = _QuadrantBase.os.path.join(_QuadrantBase.FINAL_ROOT, 'single_panels')
        _QuadrantBase.os.makedirs(final_single, exist_ok=True)
        for key, res in outputs.items():
            for ext in ['png', 'tif', 'pdf']:
                src = res[ext]
                dst = _QuadrantBase.os.path.join(final_single, _QuadrantBase.os.path.basename(src))
                _QuadrantBase.shutil.copy2(src, dst)
        final_png = _QuadrantBase.os.path.join(_QuadrantBase.FINAL_ROOT, 'FigS_quadrant_map_Ts_Rmax_original_layout_retained_without_significance_stippling_8panel_mosaic_600dpi.png')
        final_tif = _QuadrantBase.os.path.join(_QuadrantBase.FINAL_ROOT, 'FigS_quadrant_map_Ts_Rmax_original_layout_retained_without_significance_stippling_8panel_mosaic_600dpi.tif')
        final_csv = _QuadrantBase.os.path.join(_QuadrantBase.FINAL_ROOT, 'quadrant_map_Ts_Rmax_original_layout_retained_without_significance_stippling_area_statistics.csv')
        final_log = _QuadrantBase.os.path.join(_QuadrantBase.FINAL_ROOT, '24_quadrant_map_original_layout_retained_without_significance_stippling_8panel_mosaic_log.json')
        final_script = _QuadrantBase.os.path.join(_QuadrantBase.FINAL_ROOT, '24_quadrant_map_original_layout_retained_without_significance_stippling_8panel_mosaic.py')
        _QuadrantBase.shutil.copy2(stitched_png, final_png)
        _QuadrantBase.shutil.copy2(stitched_tif, final_tif)
        _QuadrantBase.shutil.copy2(stats_csv, final_csv)
        _QuadrantBase.shutil.copy2(log_path, final_log)
        _QuadrantBase.shutil.copy2(__file__, final_script)
        print(_QuadrantBase.json.dumps({'png': final_png, 'tif': final_tif, 'csv': final_csv, 'log': final_log, 'script': final_script}, ensure_ascii=False, indent=2))

class _QuadrantPlot:
    asset_path = asset_path
    module_path = module_path
    output_path = output_path
    font_path = font_path
    import importlib.util
    import json
    import os
    import shutil
    from datetime import datetime
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from PIL import Image, ImageDraw, ImageFont
    BASE_SCRIPT = module_path('four_quadrant_base.py')
    ASCII_OUT = asset_path('asset_6caec5f1c945')
    FINAL_ROOT = asset_path('asset_63b953bb2380')
    ENDPOINTS = {'pos_pos': {'light': '#9B641F', 'dark': '#3F2100'}, 'neg_pos': {'light': '#FFF176', 'dark': '#FFD21A'}, 'neg_neg': {'light': '#9BE85F', 'dark': '#2F9E00'}, 'pos_neg': {'light': '#83CFFF', 'dark': '#006FD6'}}
    BAR_ENDPOINTS = {**ENDPOINTS, 'pos_pos': {'light': '#9B641F', 'dark': '#7A4A12'}}
    TS_COLOR_LIMIT = 1.0
    RMAX_COLOR_LIMIT = 0.1
    COLOR_GAMMA = 0.7

    def to_rgb(c):
        import matplotlib.colors as mcolors
        return _QuadrantPlot.np.array(mcolors.to_rgb(c), dtype=float)

    def bivariate_cmap_fixed_ranges(X, Y, endpoints, x_limit=TS_COLOR_LIMIT, y_limit=RMAX_COLOR_LIMIT, gamma=1.0, nodata_color=(1, 1, 1)):
        valid = ~(_QuadrantPlot.np.isnan(X) | _QuadrantPlot.np.isnan(Y))
        X0 = _QuadrantPlot.np.where(valid, X, 0.0)
        Y0 = _QuadrantPlot.np.where(valid, Y, 0.0)
        Xn = _QuadrantPlot.np.clip(X0 / max(float(x_limit), 1e-12), -1, 1)
        Yn = _QuadrantPlot.np.clip(Y0 / max(float(y_limit), 1e-12), -1, 1)
        img = _QuadrantPlot.np.ones(X.shape + (3,), dtype=float)

        def paint(mask2, key):
            if not _QuadrantPlot.np.any(mask2):
                return
            strength = _QuadrantPlot.np.sqrt(Xn[mask2] ** 2 + Yn[mask2] ** 2) / _QuadrantPlot.np.sqrt(2)
            strength = _QuadrantPlot.np.clip(strength, 0, 1)
            if gamma != 1.0:
                strength = strength ** gamma
            light = _QuadrantPlot.to_rgb(endpoints[key]['light'])
            dark = _QuadrantPlot.to_rgb(endpoints[key]['dark'])
            img[mask2] = (1 - strength)[:, None] * light + strength[:, None] * dark
        paint((Xn > 0) & (Yn > 0) & valid, 'pos_pos')
        paint((Xn < 0) & (Yn > 0) & valid, 'neg_pos')
        paint((Xn < 0) & (Yn < 0) & valid, 'neg_neg')
        paint((Xn > 0) & (Yn < 0) & valid, 'pos_neg')
        img[~valid] = _QuadrantPlot.np.array(nodata_color, dtype=float)
        return _QuadrantPlot.np.clip(img, 0, 1)

    def load_base_module():
        return _QuadrantBase

    def format_lon_label(lon_value):
        if lon_value == 0:
            return '0°'
        if abs(lon_value) == 180:
            return '180°'
        if lon_value < 0:
            return f'{abs(int(lon_value))}°W'
        return f'{int(lon_value)}°E'

    def add_manual_bottom_lon_labels(ax, pathway):
        if pathway == 'SPEI':
            lon_ticks = [-180, -120, -60, 0, 60, 120]
        else:
            lon_ticks = [-120, -60, 0, 60, 120, 180]
        for lon_value in lon_ticks:
            x_frac = (lon_value + 180.0) / 360.0
            ax.text(x_frac, -0.055, _QuadrantPlot.format_lon_label(lon_value), transform=ax.transAxes, ha='center', va='top', fontsize=13, family='serif', clip_on=False, zorder=30)

    def draw_single_reference_panel(mod, pathway, season, letter, show_col_title=False, show_bottom_labels=False, comparison=False):
        lat, lon, ts, rmax, q_ts, q_r, mask = mod.read_panel(pathway, season)
        lon_raw = lon.copy()
        lon, ts = mod.sort_lon(lon_raw, ts)
        _, rmax = mod.sort_lon(lon_raw, rmax)
        _, q_ts = mod.sort_lon(lon_raw, q_ts)
        _, q_r = mod.sort_lon(lon_raw, q_r)
        _, mask = mod.sort_lon(lon_raw, mask.astype(_QuadrantPlot.np.uint8))
        mask = mask.astype(bool)
        x_slope = ts.copy()
        y_slope = rmax.copy()
        x_q = q_ts.copy()
        y_q = q_r.copy()
        x_slope[~mask] = _QuadrantPlot.np.nan
        y_slope[~mask] = _QuadrantPlot.np.nan
        x_q[~mask] = _QuadrantPlot.np.nan
        y_q[~mask] = _QuadrantPlot.np.nan
        weight_matrix = mod.create_weight_matrix(lon, lat)
        fig = _QuadrantPlot.plt.figure(figsize=(7, 4))
        ax = _QuadrantPlot.plt.axes(projection=_QuadrantPlot.ccrs.PlateCarree())
        img = _QuadrantPlot.bivariate_cmap_fixed_ranges(x_slope, y_slope, endpoints=_QuadrantPlot.ENDPOINTS, gamma=_QuadrantPlot.COLOR_GAMMA, nodata_color=(1, 1, 1))
        zero_mask = _QuadrantPlot.np.isfinite(x_slope) & _QuadrantPlot.np.isfinite(y_slope) & ((x_slope == 0) | (y_slope == 0))
        img[zero_mask] = _QuadrantPlot.to_rgb('#D9D9D9')
        dlon = abs(lon[1] - lon[0]) if len(lon) > 1 else 0.1
        dlat = abs(lat[1] - lat[0]) if len(lat) > 1 else 0.1
        ax.imshow(img, origin='lower', extent=[lon.min(), lon.max(), lat.min(), lat.max()], transform=_QuadrantPlot.ccrs.PlateCarree(), interpolation='nearest', zorder=1)
        ax.coastlines('110m', linewidth=0.8, color='black', zorder=10)
        ax.add_feature(_QuadrantPlot.cfeature.BORDERS, linewidth=0.5, color='gray', zorder=10)
        ax.add_feature(_QuadrantPlot.cfeature.LAND, facecolor='lightgray', alpha=0.2, zorder=0)
        ax.add_feature(_QuadrantPlot.cfeature.OCEAN, facecolor='lightblue', alpha=0.1, zorder=0)
        ax.set_extent([lon.min() - dlon * 0.5, lon.max() + dlon * 0.5, lat.min() - dlat * 0.5, lat.max() + dlat * 0.5], crs=_QuadrantPlot.ccrs.PlateCarree())
        gl = ax.gridlines(crs=_QuadrantPlot.ccrs.PlateCarree(), draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.bottom_labels = False
        if pathway == 'SPEI':
            gl.left_labels = True
            gl.right_labels = False
        else:
            gl.left_labels = False
            gl.right_labels = True
        gl.xlabel_style = {'size': 13, 'family': 'serif'}
        gl.ylabel_style = {'size': 13, 'family': 'serif'}
        if show_bottom_labels:
            _QuadrantPlot.add_manual_bottom_lon_labels(ax, pathway)
        ax.text(0.02, 0.98, letter, transform=ax.transAxes, ha='left', va='top', fontsize=16, fontweight='bold', family='Arial', zorder=20)
        ax.text(0.02, 0.7, season, transform=ax.transAxes, ha='left', va='center', fontsize=17, fontweight='bold', family='Times New Roman', zorder=20)
        if show_col_title:
            title = ('$R_{\\max}$' if pathway == 'SPEI' else 'Ts') if comparison else 'SPEI-SIF' if pathway == 'SPEI' else 'SSMI$_{rz}$-SIF'
            ax.text(0.5, 1.035, title, transform=ax.transAxes, ha='center', va='bottom', fontsize=12, fontweight='bold', fontstyle='italic', color='black', zorder=20)
        ax_bar, quadrant_data = mod.add_weighted_significance_bar_chart(fig, x_slope, y_slope, x_q, y_q, weight_matrix, mask, _QuadrantPlot.BAR_ENDPOINTS, position=(0.05, 0.05), width=0.18, height=0.4, show_legend=True, annotate_total=True, annotate_sig=False)
        if ax_bar is not None:
            ax_bar.set_position([0.19 if comparison else 0.232, 0.28, 0.09, 0.25])
            ax_bar.tick_params(axis='y', labelsize=9, pad=0.5)
        if not comparison:
            ax_lolli, lolli_rows = mod.add_dual_axis_lollipop_inset(fig, x_slope, y_slope, x_q, y_q, weight_matrix, mask, _QuadrantPlot.ENDPOINTS, position=(0.62, 0.06), width=0.2, height=0.16, use_weighted_mean=True, ts_range=(-1.8, 1.8), ts_tick_value=1.0, mc_tick_value=0.03, show_both_sig=False)
            if ax_lolli is not None:
                ax_lolli.set_position([0.58, 0.18, 0.2, 0.15])
        else:
            lolli_rows = []
        panel_dir = _QuadrantPlot.os.path.join(_QuadrantPlot.ASCII_OUT, '01_single_panels')
        _QuadrantPlot.os.makedirs(panel_dir, exist_ok=True)
        base = f'panel_{letter}_{pathway}_{season}'
        png = _QuadrantPlot.os.path.join(panel_dir, base + '_600dpi.png')
        tif = _QuadrantPlot.os.path.join(panel_dir, base + '_600dpi.tif')
        pdf = _QuadrantPlot.os.path.join(panel_dir, base + '.pdf')
        fig.savefig(png, dpi=600, bbox_inches='tight', pad_inches=0.035, facecolor='white', edgecolor='none', metadata={'Software': 'matplotlib'})
        fig.savefig(tif, dpi=600, bbox_inches='tight', pad_inches=0.035, facecolor='white', edgecolor='none')
        fig.savefig(pdf, bbox_inches='tight', pad_inches=0.035, facecolor='white', edgecolor='none')
        _QuadrantPlot.plt.close(fig)
        stats_rows = mod.panel_statistics(pathway, season, lat, lon, x_slope, y_slope, x_q, y_q, mask, quadrant_data, lolli_rows)
        return {'png': png, 'tif': tif, 'pdf': pdf, 'stats': stats_rows}

    def draw_bivariate_legend(mod, out_png):
        fig = _QuadrantPlot.plt.figure(figsize=(1.9, 1.8))
        ax = fig.add_axes([0.43, 0.28, 0.47, 0.56])
        norm = _QuadrantPlot.np.linspace(-1, 1, 260)
        xn, yn = _QuadrantPlot.np.meshgrid(norm, norm)
        img = _QuadrantPlot.bivariate_cmap_fixed_ranges(xn * _QuadrantPlot.TS_COLOR_LIMIT, yn * _QuadrantPlot.RMAX_COLOR_LIMIT, endpoints=_QuadrantPlot.ENDPOINTS, gamma=_QuadrantPlot.COLOR_GAMMA, nodata_color=(1, 1, 1))
        ax.imshow(img, origin='lower', extent=[-1, 1, -1, 1], interpolation='nearest', aspect='equal')
        ax.axhline(0, color='black', lw=1)
        ax.axvline(0, color='black', lw=1)
        ax.set_xticks([-1, 0, 1])
        ax.set_yticks([-1, 0, 1])
        ax.set_xticklabels(['-1', '0', '1'], fontweight='bold')
        ax.set_yticklabels(['-0.1', '0', '0.1'], fontweight='bold')
        ax.tick_params(direction='out', length=4, width=1, labelsize=12)
        ax.set_xlabel('Ts trend', fontsize=17, labelpad=6)
        ax.set_ylabel('$R_{\\max}$ trend', fontsize=17, labelpad=10)
        ax.yaxis.set_label_coords(-0.64, 0.5)
        for spine in ax.spines.values():
            spine.set_linewidth(1)
        fig.savefig(out_png, dpi=600, bbox_inches='tight', pad_inches=0.1, facecolor='white', edgecolor='none')
        _QuadrantPlot.plt.close(fig)
        return out_png

    def draw_two_row_dot_legend(out_png):
        entries = [('Ts+, $R_{\\max}$+', 'pos_pos'), ('Ts-, $R_{\\max}$+', 'neg_pos'), ('Ts-, $R_{\\max}$-', 'neg_neg'), ('Ts+, $R_{\\max}$-', 'pos_neg')]
        fig = _QuadrantPlot.plt.figure(figsize=(19.0 / 2.54, 2.85 / 2.54), dpi=150)
        ax = fig.add_axes([0, 0, 1, 1])
        ax.set_axis_off()
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        for i, (label, key) in enumerate(entries):
            xi = 0.015 + i * 0.25
            for y, shade in [(0.63, 'dark'), (0.36, 'light')]:
                ax.scatter([xi], [y], s=85, c=_QuadrantPlot.ENDPOINTS[key][shade], edgecolors='white', linewidths=0.8, clip_on=False)
                ax.text(xi + 0.025, y, label, fontsize=15.5, va='center', ha='left', clip_on=False)
        fig.savefig(out_png, dpi=600, facecolor='white', edgecolor='none', bbox_inches='tight', pad_inches=0.03)
        _QuadrantPlot.plt.close(fig)
        return out_png

    def stitch_reference_layout(panel_pngs, dot_legend_png, bivar_legend_png, out_png, out_tif):
        panels = [_QuadrantPlot.Image.open(p).convert('RGB') for p in panel_pngs]
        w = max((p.width for p in panels))
        col_gap, row_gap = (28, 10)
        left_pad, right_pad, top_pad = (8, 8, 6)
        grid_w = 2 * w + col_gap
        row_heights = [max(panels[2 * r].height, panels[2 * r + 1].height) for r in range(4)]
        grid_h = sum(row_heights) + 3 * row_gap
        positions = []
        for i, p in enumerate(panels):
            row, col = divmod(i, 2)
            positions.append((left_pad + col * (w + col_gap) + (w - p.width) // 2, top_pad + sum(row_heights[:row]) + row * row_gap + (row_heights[row] - p.height) // 2))

        def frame_x(im):
            a = _QuadrantPlot.np.asarray(im)
            counts = (a.max(axis=2) < 85).sum(axis=0)
            columns = _QuadrantPlot.np.flatnonzero(counts > 0.7 * counts.max())
            assert len(columns) > 1, 'Cannot locate map frame.'
            return (int(columns[0]), int(columns[-1]))
        xl, _ = frame_x(panels[6])
        _, xr = frame_x(panels[7])
        legend_left = positions[6][0] + xl
        legend_right = positions[7][0] + xr
        left_frame_left, left_frame_right = frame_x(panels[6])
        right_frame_left, right_frame_right = frame_x(panels[7])
        from io import BytesIO
        from matplotlib.mathtext import math_to_image
        from matplotlib.font_manager import FontProperties
        heading_font = FontProperties(family='Arial', size=15.5 * 1.05)
        header_images = []
        for formula in ['$\\mathrm{Ts}:$', '$R_{\\max}:$']:
            buffer = BytesIO()
            math_to_image(formula, buffer, prop=heading_font, dpi=600, format='png', color='black')
            buffer.seek(0)
            header_images.append(_QuadrantPlot.Image.open(buffer).convert('RGBA'))
        circular_block_left = positions[6][0] + left_frame_left + (left_frame_right - left_frame_left) / 36
        header_label_left = circular_block_left
        header_width = max((im.width for im in header_images))
        circle_left = positions[6][0] + left_frame_left + (left_frame_right - left_frame_left) / 4
        circle_right = positions[7][0] + right_frame_left + (right_frame_right - right_frame_left) * 2 / 3
        available = circle_right - circle_left
        dot = _QuadrantPlot.Image.open(dot_legend_png).convert('RGB')
        a = _QuadrantPlot.np.asarray(dot)
        colored = (a.max(axis=2) - a.min(axis=2) > 25) & (a.min(axis=2) < 235)
        cols = _QuadrantPlot.np.flatnonzero(colored.any(axis=0))
        breaks = _QuadrantPlot.np.flatnonzero(_QuadrantPlot.np.diff(cols) > 1)
        starts = _QuadrantPlot.np.r_[cols[0], cols[breaks + 1]]
        assert len(starts) == 4
        ink_right = _QuadrantPlot.np.flatnonzero((a.min(axis=2) < 250).any(axis=0))[-1] + 1
        entries = []
        for i, start in enumerate(starts):
            right = int(starts[i + 1] - 3) if i < 3 else int(ink_right + 2)
            entry = dot.crop((int(start) - 2, 0, right, dot.height))
            ink = _QuadrantPlot.np.asarray(entry).min(axis=2) < 250
            last = _QuadrantPlot.np.flatnonzero(ink.any(axis=0))[-1]
            entries.append(entry.crop((0, 0, min(entry.width, int(last) + 4), entry.height)))
        biv = _QuadrantPlot.Image.open(bivar_legend_png).convert('RGB')
        target_h = int(dot.height * 0.92 * 1.06)
        biv = biv.resize((int(biv.width * target_h / biv.height), target_h), _QuadrantPlot.Image.Resampling.LANCZOS)
        segments = entries
        minimum_gap = 95
        scale = min(1.05, (available - 3 * minimum_gap) / sum((e.width for e in segments)))
        segments = [e.resize((round(e.width * scale), round(e.height * scale)), _QuadrantPlot.Image.Resampling.LANCZOS) for e in segments]
        gap = (available - sum((e.width for e in segments))) / 3
        assert gap >= minimum_gap - 2
        biv = biv.resize((round(biv.width * 1.05), round(biv.height * 1.05)), _QuadrantPlot.Image.Resampling.LANCZOS)
        footer_h = max([e.height for e in segments] + [biv.height])
        footer_y = top_pad + grid_h + 12
        canvas = _QuadrantPlot.Image.new('RGB', (left_pad + grid_w + right_pad, footer_y + footer_h + 16), 'white')
        for im, xy in zip(panels, positions):
            canvas.paste(im, xy)
        draw = _QuadrantPlot.ImageDraw.Draw(canvas)
        font = _QuadrantPlot.ImageFont.truetype(_QuadrantPlot.font_path('arial.ttf'), 108)
        lb = positions[6]
        rb = positions[7]
        cx = int((lb[0] + panels[6].width + rb[0]) / 2)
        cy = int(max(lb[1] + panels[6].height, rb[1] + panels[7].height) - 72)
        draw.text((cx, cy), '180°', fill='black', font=font, anchor='mm')
        x = float(circle_left)
        layout = []
        for i, entry in enumerate(segments):
            px = int(round(x))
            py = footer_y + (footer_h - entry.height) // 2 - 80
            canvas.paste(entry, (px, py), entry.convert('L').point(lambda value: 0 if value == 255 else 255))
            layout.append({'group': i + 1, 'x': px, 'y': py, 'width': entry.width, 'height': entry.height})
            x += entry.width + gap
        assert abs(layout[-1]['x'] + layout[-1]['width'] - circle_right) <= 1
        biv_x = legend_right - biv.width
        assert circle_right + 40 < biv_x, 'Circular and bivariate legends must remain separated.'
        canvas.paste(biv, (int(biv_x), footer_y + (footer_h - biv.height) // 2))
        pixels = _QuadrantPlot.np.asarray(segments[0])
        chroma = (pixels.max(axis=2) - pixels.min(axis=2) > 25) & (pixels.min(axis=2) < 235)
        marker_rows = _QuadrantPlot.np.flatnonzero(chroma.any(axis=1))
        row_breaks = _QuadrantPlot.np.flatnonzero(_QuadrantPlot.np.diff(marker_rows) > 1)
        assert len(row_breaks) == 1
        top_rows = marker_rows[:row_breaks[0] + 1]
        bottom_rows = marker_rows[row_breaks[0] + 1:]
        header_y = [layout[0]['y'] + float(top_rows.mean()), layout[0]['y'] + float(bottom_rows.mean())]
        header_records = []
        for text, image, yy in zip(['Ts:', 'Rmax:'], header_images, header_y):
            hx = int(round(header_label_left))
            hy = int(round(yy - image.height / 2))
            canvas.paste(image, (hx, hy), image)
            header_records.append({'text': text, 'left': hx, 'center_y': round(yy), 'width': image.width, 'height': image.height, 'max_is_subscript': text == 'Rmax:'})
        assert abs(header_records[0]['left'] - header_label_left) <= 1
        canvas.save(out_png, dpi=(600, 600))
        canvas.save(out_tif, dpi=(600, 600))
        with open(_QuadrantPlot.os.path.join(_QuadrantPlot.ASCII_OUT, 'footer_layout_validation.json'), 'w', encoding='utf-8') as f:
            _QuadrantPlot.json.dump({'legend_left_map_180': legend_left, 'legend_right_map_180': legend_right, 'bar_axes_original': [0.232, 0.28, 0.09, 0.25], 'grey_circular_entry': False, 'bar_zero_label': 'either = 0', 'row_headers': header_records, 'header_font_px': round(15.5 * 600 / 72 * 1.05), 'horizontal_gap_px': gap, 'circular_block_left_170W': circular_block_left, 'row_header_left_170W': header_label_left, 'circular_raise_px': 80, 'four_groups_left_90W': circle_left, 'first_circle_left': circle_left, 'circle_right_60E': circle_right, 'bivariate_x': biv_x, 'segments': layout}, f, indent=2)
        return (out_png, out_tif)

    def main():
        _QuadrantPlot.os.makedirs(_QuadrantPlot.ASCII_OUT, exist_ok=True)
        _QuadrantPlot.os.makedirs(_QuadrantPlot.FINAL_ROOT, exist_ok=True)
        mod = _QuadrantPlot.load_base_module()
        mod.ASCII_OUT = _QuadrantPlot.ASCII_OUT
        mod.FINAL_ROOT = _QuadrantPlot.FINAL_ROOT
        mod.ENDPOINTS.clear()
        mod.ENDPOINTS.update(_QuadrantPlot.ENDPOINTS)
        mod.setup_reference_style()
        seasons = ['MAM', 'JJA', 'SON', 'DJF']
        panel_plan = []
        for i, season in enumerate(seasons):
            show_bottom = season == 'DJF'
            panel_plan.append(('SPEI', season, chr(ord('a') + i), i == 0, show_bottom))
            panel_plan.append(('SSMIrz', season, chr(ord('e') + i), i == 0, show_bottom))
        outputs = {}
        stats_rows = []
        panel_pngs = []
        for pathway, season, letter, show_title, show_bottom in panel_plan:
            res = _QuadrantPlot.draw_single_reference_panel(mod, pathway, season, letter, show_title, show_bottom)
            outputs[f'{letter}_{pathway}_{season}'] = {k: v for k, v in res.items() if k != 'stats'}
            stats_rows.extend(res['stats'])
            panel_pngs.append(res['png'])
        parts_dir = _QuadrantPlot.os.path.join(_QuadrantPlot.ASCII_OUT, '02_legends')
        _QuadrantPlot.os.makedirs(parts_dir, exist_ok=True)
        dot_legend = _QuadrantPlot.draw_two_row_dot_legend(_QuadrantPlot.os.path.join(parts_dir, 'two_row_dot_legend_600dpi.png'))
        bivar_legend = _QuadrantPlot.draw_bivariate_legend(mod, _QuadrantPlot.os.path.join(parts_dir, 'bivariate_legend_Ts_Rmax_600dpi.png'))
        stitched_png = _QuadrantPlot.os.path.join(_QuadrantPlot.ASCII_OUT, 'FigS_fourcolor_Ts_Rmax_reference_layout_parts_600dpi.png')
        stitched_tif = _QuadrantPlot.os.path.join(_QuadrantPlot.ASCII_OUT, 'FigS_fourcolor_Ts_Rmax_reference_layout_parts_600dpi.tif')
        _QuadrantPlot.stitch_reference_layout(panel_pngs, dot_legend, bivar_legend, stitched_png, stitched_tif)
        stats_csv = _QuadrantPlot.os.path.join(_QuadrantPlot.ASCII_OUT, 'fourcolor_reference_layout_area_stats.csv')
        _QuadrantPlot.pd.DataFrame(stats_rows).to_csv(stats_csv, index=False, encoding='utf-8-sig')
        log_path = _QuadrantPlot.os.path.join(_QuadrantPlot.ASCII_OUT, '32_quadrant_map_reference_layout_panel_mosaic_log.json')
        with open(log_path, 'w', encoding='utf-8') as f:
            _QuadrantPlot.json.dump({'created_at': _QuadrantPlot.datetime.now().isoformat(timespec='seconds'), 'layout': 'Reference Spring notebook style: panels, two-row dot legend, and bivariate legend are generated separately and stitched.', 'panel_plan': [f'{letter}_{pathway}_{season}' for pathway, season, letter, _, _ in panel_plan], 'no_significance_stippling': True, 'units': {'Ts trend': 'months year-1', 'Rmax trend': 'Spearman r year-1'}, 'color_scaling': {'Ts trend fixed range': [-_QuadrantPlot.TS_COLOR_LIMIT, _QuadrantPlot.TS_COLOR_LIMIT], 'Rmax trend fixed range': [-_QuadrantPlot.RMAX_COLOR_LIMIT, _QuadrantPlot.RMAX_COLOR_LIMIT], 'gamma': _QuadrantPlot.COLOR_GAMMA}, 'color_mapping': {'Ts+ Rmax+': _QuadrantPlot.ENDPOINTS['pos_pos'], 'Ts- Rmax+': _QuadrantPlot.ENDPOINTS['neg_pos'], 'Ts- Rmax-': _QuadrantPlot.ENDPOINTS['neg_neg'], 'Ts+ Rmax-': _QuadrantPlot.ENDPOINTS['pos_neg']}, 'outputs': outputs, 'legends': {'dot_legend': dot_legend, 'bivariate_legend': bivar_legend}, 'stitched': {'png': stitched_png, 'tif': stitched_tif}, 'stats_csv': stats_csv}, f, ensure_ascii=False, indent=2)
        final_panels = _QuadrantPlot.os.path.join(_QuadrantPlot.FINAL_ROOT, '01_individual_inset')
        final_legends = _QuadrantPlot.os.path.join(_QuadrantPlot.FINAL_ROOT, '02_legend')
        _QuadrantPlot.os.makedirs(final_panels, exist_ok=True)
        _QuadrantPlot.os.makedirs(final_legends, exist_ok=True)
        for res in outputs.values():
            for ext in ['png', 'tif', 'pdf']:
                _QuadrantPlot.shutil.copy2(res[ext], _QuadrantPlot.os.path.join(final_panels, _QuadrantPlot.os.path.basename(res[ext])))
        final_dot = _QuadrantPlot.os.path.join(final_legends, 'two_row_marker_legend_600dpi.png')
        final_bivar = _QuadrantPlot.os.path.join(final_legends, 'bivariate_gradient_colorbar_Ts_Rmax_600dpi.png')
        final_png = _QuadrantPlot.os.path.join(_QuadrantPlot.FINAL_ROOT, 'FigS_quadrant_map_Ts_Rmax_reference_layout_panel_mosaic_without_significance_stippling_600dpi.png')
        final_tif = _QuadrantPlot.os.path.join(_QuadrantPlot.FINAL_ROOT, 'FigS_quadrant_map_Ts_Rmax_reference_layout_panel_mosaic_without_significance_stippling_600dpi.tif')
        final_csv = _QuadrantPlot.os.path.join(_QuadrantPlot.FINAL_ROOT, 'quadrant_map_Ts_Rmax_reference_layout_area_statistics.csv')
        final_log = _QuadrantPlot.os.path.join(_QuadrantPlot.FINAL_ROOT, '32_quadrant_map_reference_layout_panel_mosaic_log.json')
        final_script = _QuadrantPlot.os.path.join(_QuadrantPlot.FINAL_ROOT, 'four_quadrant_layout.py')
        _QuadrantPlot.shutil.copy2(dot_legend, final_dot)
        _QuadrantPlot.shutil.copy2(bivar_legend, final_bivar)
        _QuadrantPlot.shutil.copy2(stitched_png, final_png)
        _QuadrantPlot.shutil.copy2(stitched_tif, final_tif)
        _QuadrantPlot.shutil.copy2(stats_csv, final_csv)
        _QuadrantPlot.shutil.copy2(log_path, final_log)
        _QuadrantPlot.shutil.copy2(__file__, final_script)
        print(_QuadrantPlot.json.dumps({'final_png': final_png, 'final_tif': final_tif, 'single_panels': final_panels, 'legends': final_legends, 'stats': final_csv, 'log': final_log, 'script': final_script}, ensure_ascii=False, indent=2))

def load(name):
    return {'four_quadrant_base': _QuadrantBase, 'four_quadrant_layout': _QuadrantPlot}[Path(name).stem]

In [ ]:
from pathlib import Path
from functools import lru_cache
import netCDF4 as nc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from PIL import Image, ImageDraw, ImageFont
OUT = OUTPUT / 'Fig.S1'
OUT.mkdir(parents=True, exist_ok=True)
COLORS = {1: '#FAD532', 2: '#269CFD', 3: '#FF5E00', 4: '#3A0070'}
ORDER = [1, 2, 4, 3]
LABELS = {1: '++', 2: '--', 4: '+-', 3: '-+'}
SEASONS = ['MAM']

@lru_cache(maxsize=16)
def read(metric, pathway, season):
    path = INPUT_DIR / f'{pathway}_{metric}.nc'
    with open_nc('trend.nc', memory=path.read_bytes()) as dataset:
        names = nc.chartostring(dataset['season_name'][:]).tolist() if 'season_name' in dataset.variables else SEASONS
        index = names.index(season)
        values = lambda key: np.asarray(np.ma.filled(dataset[key][:].astype(float), np.nan))
        lat, lon = (values('lat'), values('lon'))
        slope = values('slope')[index]
        valid = (values('analysis_mask')[index] == 1) & np.isfinite(slope)
    return (lat, lon, slope, valid)

def read_panel(column, season):
    metric = 'Rmax' if column == 'SPEI' else 'Ts'
    lat, lon, a, va = read(metric, 'SPEI', season)
    lat_b, lon_b, b, vb = read(metric, 'SSMIrz', season)
    assert np.array_equal(lat, lat_b) and np.array_equal(lon, lon_b)
    q = np.full(a.shape, np.nan)
    return (lat.copy(), lon.copy(), a.copy(), b.copy(), q.copy(), q.copy(), va & vb)

def add_bar(fig, a, b, q_a, q_b, weights, common, endpoints, **kwargs):
    denom = weights[common & np.isfinite(a) & np.isfinite(b)].sum()
    masks = {1: (a > 0) & (b > 0), 2: (a < 0) & (b < 0), 4: (a > 0) & (b < 0), 3: (a < 0) & (b > 0)}
    values = [float(100 * weights[masks[i] & common].sum() / denom) for i in ORDER]
    ax = fig.add_axes([0.19, 0.28, 0.09, 0.25])
    ax.barh(np.arange(4), values, color=[COLORS[i] for i in ORDER], alpha=0.95, edgecolor='white', linewidth=1)
    ax.set_yticks(np.arange(4), labels=[LABELS[i] for i in ORDER], fontsize=9)
    ax.set_xlim(0, 40)
    ax.set_xticks([0, 40])
    ax.invert_yaxis()
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=9)
    for y, value in enumerate(values):
        ax.text(value + 0.8, y, f'{value:.1f}%', ha='left', va='center', fontsize=8.5, weight='bold', color='0.1')
    for edge in ['top', 'right']:
        ax.spines[edge].set_visible(False)
    ax.grid(axis='x', alpha=0.25, linestyle='--')
    return (ax, [])

def sign_image(a, b, endpoints=None, **kwargs):
    image = np.ones(a.shape + (3,))
    classes = {1: (a > 0) & (b > 0), 2: (a < 0) & (b < 0), 3: (a < 0) & (b > 0), 4: (a > 0) & (b < 0)}
    for key, mask in classes.items():
        image[mask] = to_rgb(COLORS[key])
    return image

def stitch_descriptive_layout(panel_pngs, legend_png, out_png, out_tif):
    panels = [Image.open(p).convert('RGB') for p in panel_pngs]
    w = max((p.width for p in panels))
    col_gap = 28
    row_gap = 10
    row_heights = [max(panels[2 * r].height, panels[2 * r + 1].height) for r in range(len(panel_pngs) // 2)]
    grid_w = 2 * w + col_gap
    grid_h = sum(row_heights) + 3 * row_gap
    legend = Image.open(legend_png).convert('RGB')
    canvas_w = 8 + grid_w + 8
    if legend.width != canvas_w:
        legend = legend.resize((canvas_w, round(legend.height * canvas_w / legend.width)), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', (canvas_w, 6 + grid_h + legend.height), 'white')
    positions = []
    for idx, im in enumerate(panels):
        row, col = divmod(idx, 2)
        x = 8 + col * (w + col_gap) + (w - im.width) // 2
        y = 6 + sum(row_heights[:row]) + row * row_gap + (row_heights[row] - im.height) // 2
        canvas.paste(im, (x, y))
        positions.append((x, y, im.width, im.height))
    left, right = (positions[-2], positions[-1])
    draw = ImageDraw.Draw(canvas)
    font = ImageFont.truetype(font_path('arial.ttf'), 108)
    draw.text((int((left[0] + left[2] + right[0]) / 2), int(max(left[1] + left[3], right[1] + right[3]) - 72)), '180°', fill='black', font=font, anchor='mm')
    canvas.paste(legend, (0, 6 + grid_h))
    canvas.save(out_png, dpi=(600, 600))
    canvas.save(out_tif, dpi=(600, 600))
base = load('four_quadrant_base.py')
plot = load('four_quadrant_layout.py')
base.setup_reference_style()
base.read_panel = read_panel
base.add_weighted_significance_bar_chart = add_bar
base.panel_statistics = lambda *args, **kwargs: []
plot.bivariate_cmap_fixed_ranges = sign_image
plot.ASCII_OUT = str(OUT)
plot.load_base_module = lambda: base
panels = []
statistics = []
for i, season in enumerate(SEASONS):
    for column, letter in [('SPEI', chr(97 + i)), ('SSMIrz', chr(101 + i))]:
        lat, lon, a, b, qa, qb, valid = read_panel(column, season)
        weights = np.broadcast_to(np.cos(np.deg2rad(lat))[:, None], valid.shape)
        denominator = weights[valid].sum()
        row = {'metric': 'Rmax' if column == 'SPEI' else 'Ts', 'season': season}
        for label, selected in [('++', (a > 0) & (b > 0)), ('--', (a < 0) & (b < 0)), ('+-', (a > 0) & (b < 0)), ('-+', (a < 0) & (b > 0)), ('either_zero', (a == 0) | (b == 0))]:
            row[label] = 100 * weights[valid & selected].sum() / denominator
        assert np.isclose(sum((value for key, value in row.items() if key not in {'metric', 'season'})), 100)
        statistics.append(row)
        panel = plot.draw_single_reference_panel(base, column, season, letter, show_col_title=i == 0, show_bottom_labels=i == 3, comparison=True)
        panels.append(panel['png'])
pd.DataFrame(statistics).to_csv(OUT / 'area_shares.csv', index=False)
legend = OUT / 'legend_600dpi.png'
fig = plt.figure(figsize=(12.0066666667, 1.2), dpi=600)
texts = {1: '++ Both increase', 2: '-- Both decrease', 4: '+- SPEI-SIF increases, SSMI$_{rz}$-SIF decreases', 3: '-+ SPEI-SIF decreases, SSMI$_{rz}$-SIF increases'}
handles = [Line2D([], [], marker='o', linestyle='', color=COLORS[key], markersize=8, label=texts[key]) for key in [1, 4, 2, 3]]
fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.99), ncol=2, frameon=False, fontsize=11, columnspacing=2, handletextpad=0.5, labelspacing=0.8)
fig.text(0.5, 0.3, 'First sign: SPEI-SIF. Second sign: SSMI$_{rz}$-SIF.', ha='center', va='center', fontsize=10)
fig.text(0.5, 0.08, 'Area shares use jointly valid area, including zero trends.', ha='center', va='center', fontsize=9)
fig.savefig(legend, dpi=600, facecolor='white')
plt.close(fig)
stitch_descriptive_layout(panels, str(legend), str(OUT / 'Fig.S1_600dpi.png'), str(OUT / 'Fig.S1_600dpi.tif'))